# 02 - Price model

XGBoost regression on the scraped listings. Target is price per m2 rather than total price, so
floor area does not dominate.

Data: 4,833 listings, 120 numeric features after encoding. Mean price per m2 is 77.29.

Validation: `RepeatedKFold`, 10 splits x 3 repeats, scored on mean absolute error.

Result: MAE 10.77 (sd 0.53), about 13.9% of the mean.

Parameters are XGBoost defaults: `learning_rate=0.3`, `max_depth=6`, `n_estimators=100`,
`subsample=1`, `colsample_bytree=1`, `tree_method='exact'`. Notebook 03 runs Optuna, GridSearchCV
and RandomizedSearchCV against this dataset. None of them beat the defaults, which is why the
defaults are used here.


In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold

In [2]:
df = pd.read_excel(r'data/dataset.xlsx')

In [3]:
X=df.drop(columns=['price','no'])

In [4]:
y=df["price"]

In [5]:
model = xgb.XGBRegressor() 

In [6]:
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

In [7]:
scores = cross_val_score(model, X, y, 
         scoring='neg_mean_absolute_error',
         cv=cv, n_jobs=-1,error_score='raise')

In [8]:
scores = np.absolute(scores)
print('Mean MAE: %.3f (%.3f)' % (scores.mean(), scores.std()) )

Mean MAE: 10.769 (0.531)


In [16]:
model.fit(X,y)

XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, enable_categorical=False,
             gamma=0, gpu_id=-1, importance_type=None,
             interaction_constraints='', learning_rate=0.300000012,
             max_delta_step=0, max_depth=6, min_child_weight=1, missing=nan,
             monotone_constraints='()', n_estimators=100, n_jobs=8,
             num_parallel_tree=1, predictor='auto', random_state=0, reg_alpha=0,
             reg_lambda=1, scale_pos_weight=1, subsample=1, tree_method='exact',
             validate_parameters=1, verbosity=None)

In [20]:
predict_input = pd.read_excel(r'data/predict_input.xlsx')

In [21]:
yhat=model.predict(predict_input)

In [22]:
print('Predicted: %.3f' % yhat)

Predicted: 53.762
